# D-FINE-cpp Python quickstart

Zero-setup path to real-time object detection: the `dfine` wheel bundles the C++/TensorRT
runtime (`libdfine.so`, sm_89 / linux_x86_64) and an engine-build script; the release ships
the production ONNX. Everything below runs in a fresh venv — no repo checkout, no OpenCV,
no compiler.

Requirements: Linux x86_64, an NVIDIA GPU with a CUDA-12 driver. On GPUs other than
Ada (sm_89), build `libdfine.so` from source instead (`./build.sh && pip install -e python/`) —
the rest of the notebook is identical.

In [ ]:
%pip install "dfine[cli] @ https://github.com/PogChamper/dfine-cpp/releases/download/v0.3.2/dfine-0.3.2-py3-none-linux_x86_64.whl" "tensorrt-cu12==10.13.*" pillow

## 1. Get the production ONNX and compile the engine

TensorRT engines are GPU- and TRT-version-specific, so the engine is compiled locally
(one-time, ~1-3 min). `dfine_m_slim` is the surgical-FP16 build — lossless COCO full-val
mAP vs FP32, ~2.3x its throughput.

In [ ]:
!curl -sLO https://github.com/PogChamper/dfine-cpp/releases/download/v0.3.2/dfine_m_slim.onnx \
      -sLO https://github.com/PogChamper/dfine-cpp/releases/download/v0.3.2/dfine_m_slim.json
!dfine build --model m --onnx dfine_m_slim.onnx --output dfine_m_slim.engine

## 2. Detect

In [ ]:
import urllib.request

import numpy as np
from PIL import Image

from dfine import Detector

url = "http://images.cocodataset.org/val2017/000000039769.jpg"  # two cats on a couch
img = Image.open(urllib.request.urlopen(url)).convert("RGB")
frame = np.asarray(img)                       # HWC uint8, RGB

det = Detector("dfine_m_slim.engine", threshold=0.5)
detections = det.detect(frame)
for d in detections:
    print(f"{d.class_name:12s} {d.score:.3f}  {tuple(round(v, 1) for v in d.box.as_tuple())}")

In [ ]:
from PIL import ImageDraw

vis = img.copy()
draw = ImageDraw.Draw(vis)
for d in detections:
    x1, y1, x2, y2 = d.box.as_tuple()
    draw.rectangle([x1, y1, x2, y2], outline=(255, 80, 80), width=3)
    draw.text((x1 + 3, max(0, y1 - 12)), f"{d.class_name} {d.score:.2f}", fill=(255, 80, 80))
vis

## 3. How fast is it here?

End-to-end (CUDA preprocess + TensorRT + decode) on *your* GPU. Reference: the surgical
tier measures 288 img/s b1 / 526 b8 on an RTX 4070 Ti SUPER (this slim build benched +2% on
top at b8); PyTorch on the same card does 31 / 66.

In [ ]:
import time

for _ in range(20):                                   # warmup
    det.detect(frame)

n = 200
t0 = time.perf_counter()
for _ in range(n):
    det.detect(frame)
b1 = n / (time.perf_counter() - t0)

batch = [frame] * 8                                   # engine was built with --max-batch 8
for _ in range(5):
    det.detect_batch(batch)
t0 = time.perf_counter()
for _ in range(n // 8):
    det.detect_batch(batch)
b8 = (n // 8) * 8 / (time.perf_counter() - t0)

print(f"batch-1: {b1:6.0f} img/s   ({1000 / b1:.2f} ms/frame)")
print(f"batch-8: {b8:6.0f} img/s   ({1000 / b8 * 8:.2f} ms/batch)")

## Where to go next

- **Speed sliders** — the release also documents export-time accuracy/speed trade-offs
  (`--num-queries`, `--cascade`, `--eval-idx`): up to **1556 img/s** on D-FINE-N and
  **686 img/s** on M. Every measured point:
  [docs/RESEARCH_MATRIX.md](https://github.com/PogChamper/dfine-cpp/blob/main/docs/RESEARCH_MATRIX.md).
- **Video / streaming** — `gpu_decode=True, full_pipeline_graph=True` + `det.freeze(...)`
  collapses the whole frame pipeline into one `cudaGraphLaunch` (2.47 ms e2e b1 on M):
  see [python/README.md](https://github.com/PogChamper/dfine-cpp/blob/main/python/README.md).
- **Your own weights** — `dfine export` / `export_dfine_onnx.py --opset 19` +
  `convert_fp16_surgical.py --slim`, then this notebook unchanged (pass
  `class_names=[...]` to `Detector` for non-COCO label sets).